In [1]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam


In [21]:
def csv2df(dir_path):
    train_data_frames, test_data_frames = [], []
    testing_files = ["continuous", "flooding", "normal", "plateau", "playback", "suppress"]
    csv_path_train = dir_path + "/train_1.csv" #first one is structured different than the rest
    df_temp = pd.read_csv(csv_path_train)
    train_data_frames.append(df_temp)
    for i in range(2, 5):
        csv_path_train = dir_path + "/train_" + str(i) + ".csv"
        df_temp = pd.read_csv(csv_path_train, header=None, names=["Label",  "Time", "ID", "Signal1",  "Signal2",  "Signal3",  "Signal4"])
        train_data_frames.append(df_temp)
    for file_suffix in testing_files:
        csv_path_test = dir_path + "/test_" + file_suffix + ".csv"
        df_temp = pd.read_csv(csv_path_test, low_memory=False, names=["Label",  "Time", "ID", "Signal1",  "Signal2",  "Signal3",  "Signal4"])
        test_data_frames.append(df_temp)
    test_df = pd.concat(test_data_frames)
    train_df = pd.concat(train_data_frames)
    return train_df, test_df

In [ ]:
def combine_balance_split(train_data, test_data):
    # Combine train + test
    data = pd.concat([train_data, test_data], ignore_index=True)
    # Split into features and labels
    X = data.drop(columns=["Label"])
    y = data["Label"]
    # undersample normal traffic
    normal = data[data["Label"] == 0]
    attack = data[data["Label"] == 1]
    # Match counts
    normal_downsampled = resample(normal, #ERROR HERE 
                                replace=False,
                                n_samples=len(attack),
                                random_state=42)
    balanced = pd.concat([normal_downsampled, attack])
    X_bal = balanced.drop(columns=["Label"])
    y_bal = balanced["Label"]
    X_train, X_val, y_train, y_val = train_test_split(
        X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
    )
    return X_train, X_val, y_train, y_val

In [24]:

# Load all training (nromal) and test (normal + attack) CSVs 
train_data, test_data = csv2df("./SynCAN-DATA")


In [27]:

X_train, X_val, y_train, y_val = combine_balance_split(train_data, test_data)


InvalidParameterError: The 'n_samples' parameter of resample must be an int in the range [1, inf) or None. Got 0 instead.

In [19]:
np.savez_compressed(
        "SynCANCleanLabeled.npz",
        X_train=X_train,
        y_train=y_train,
        X_test=X_val,
        y_test=y_val
    )   

NameError: name 'X_train' is not defined

In [ ]:

# 7. Simple feedforward NN (like INDRA notebook style)
# model = Sequential([
#     Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
#     Dropout(0.3),
#     Dense(64, activation="relu"),
#     Dropout(0.3),
#     Dense(1, activation="sigmoid")
# ])

# model.compile(optimizer=Adam(learning_rate=1e-3),
#               loss="binary_crossentropy",
#               metrics=["accuracy"])


In [ ]:

# 8. Train
# history = model.fit(X_train, y_train,
#                     validation_data=(X_val, y_val),
#                     epochs=10,
#                     batch_size=256)


In [ ]:

# # 9. Evaluate
# loss, acc = model.evaluate(X_val, y_val)
# print(f"Validation accuracy: {acc:.3f}")